# CompuOpti: multi-objective staff scheduling with OR-Tools CP-SAT

Decision Modelling project (CentraleSupélec). Everything is in this notebook: data, an independent
checker, the CP-SAT model, two exact Pareto methods, plots in objective and decision space, and an
instance generator.

**Run:** `uv run jupyter lab CompuOpti_staffing.ipynb`, then *Run all*. With the default budgets the
notebook takes about 15 minutes. Raise `BUDGET` to get more medium/large points proven.

In [1]:
import contextlib
import itertools
import json
import random
import time
from collections import defaultdict
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import interact
from ortools.sat.python import cp_model

TL, BUDGET, WORKERS = 10, 120, 8  # seconds per solve, seconds per Pareto front, CPU threads
OBJS = ["profit", "duration", "projects_per_staff", "late_jobs", "team_size", "imbalance"]
BRIEF = OBJS[:4]  # the brief's four objectives; the last two are our extras
BLUE, LIGHT, ORANGE, GREY = "#2a78d6", "#9ec5f4", "#ec835a", "#e1e0d9"

## 1. Data, read as a real staffing problem

* Each employee-day goes to one (project, qualification) pair, or stays idle.
* **A rejected project gets no work at all.** Nobody half-staffs a project that won't be delivered,
  so coverage is an equality: a project is either fully staffed or untouched.
* **Duration is the calendar span** from a project's first to last working day, including holiday
  gaps, because that is the wait the client sees.
* **Only delivered projects can be late.** A dropped project counts as declined, not late.
* All three instances need more person-days than the staff have (demand/capacity > 1), so the real
  decision is *which projects to accept*.

In [2]:
def prepare(d, name="generated"):
    """Index staff/jobs by name and list the legal (staff, job, qual, day) slots."""
    d["name"], d["days"] = name, range(1, d["horizon"] + 1)
    d["S"] = {s["name"]: s for s in d["staff"]}
    d["J"] = {j["name"]: j for j in d["jobs"]}
    for j in d["jobs"]:
        j["req"] = j["working_days_per_qualification"]
    d["slots"] = [
        (s["name"], j["name"], q, t)
        for s in d["staff"]
        for j in d["jobs"]
        for q in j["req"]
        if q in s["qualifications"]
        for t in d["days"]
        if t not in s["vacations"]
    ]
    return d

In [3]:
def stats(I):
    cap = sum(I["horizon"] - len(s["vacations"]) for s in I["staff"])
    demand = sum(sum(j["req"].values()) for j in I["jobs"])
    return {
        "days": I["horizon"],
        "staff": len(I["staff"]),
        "jobs": len(I["jobs"]),
        "x variables": len(I["slots"]),
        "demand / capacity": round(demand / cap, 2),
    }


INST = {
    n: prepare(json.loads(Path(f"Datasets/{n}_instance.json").read_text()), n)
    for n in ["toy", "medium", "large"]
}
pd.DataFrame({n: stats(I) for n, I in INST.items()})

,toy,medium,large
days,5.00,22.00,36.00
staff,3.00,5.00,6.00
jobs,5.00,15.00,25.00
x variables,90.00,1298.00,3939.00
demand / capacity,1.08,1.32,1.41


## 2. Independent checker

A schedule is a dict `{(employee, day): (job, qualification)}`, so "one task per day" holds by
construction. Every number in this notebook comes from `evaluate`, never from the solver's own
variables. That way a modelling mistake can't hide.

In [4]:
def utilisation(I, sched):
    """% of available days worked, for staff who could work on at least one job."""
    load = defaultdict(int)
    for i, _ in sched:
        load[i] += 1
    needed = {q for j in I["jobs"] for q in j["req"]}
    return {
        s["name"]: 100 * load[s["name"]] // (I["horizon"] - len(s["vacations"]))
        for s in I["staff"]
        if needed & set(s["qualifications"]) and len(s["vacations"]) < I["horizon"]
    }

In [5]:
def evaluate(I, sched):
    """Check every hard rule, then compute the six objectives."""
    work = defaultdict(lambda: defaultdict(int))
    days, team, jobs_of = defaultdict(set), defaultdict(set), defaultdict(set)
    for (i, t), (j, q) in sched.items():
        s = I["S"][i]
        assert q in s["qualifications"] and q in I["J"][j]["req"], "skill rule"
        assert t in I["days"] and t not in s["vacations"], "holiday rule"
        work[j][q] += 1
        days[j].add(t)
        team[j].add(i)
        jobs_of[i].add(j)
    assert all(work[j] == I["J"][j]["req"] for j in work), "a job is partly or over-staffed"
    late = {j: max(0, max(days[j]) - I["J"][j]["due_date"]) for j in work}
    util = utilisation(I, sched).values()
    return {
        "profit": sum(I["J"][j]["gain"] - I["J"][j]["daily_penalty"] * late[j] for j in work),
        "duration": max((max(d) - min(d) + 1 for d in days.values()), default=0),
        "projects_per_staff": max(map(len, jobs_of.values()), default=0),
        "late_jobs": sum(v > 0 for v in late.values()),
        "team_size": max(map(len, team.values()), default=0),
        "imbalance": max(util, default=0) - min(util, default=0),
    }

In [6]:
evaluate(INST["toy"], {})  # doing nothing: feasible, every objective 0

{'profit': 0,
 'duration': 0,
 'projects_per_staff': 0,
 'late_jobs': 0,
 'team_size': 0,
 'imbalance': 0}